In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix,accuracy_score,precision_score,
recall_score,f1_score,roc_auc_score)

df=pd.read_csv(r"C:\Users\sinan\OneDrive\Desktop\5TH-sem_ojt\project1_diabetes\data\processed\hiba_p1w2_features.csv")
feature_cols = ["age","pregnancies","glucose","blood_pressure",
 "skin_thickness","insulin","bmi","diabetes_pedigree",
              "bmi_category","glucose_category"]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(
    x,y,test_size=0.2,random_state=42,stratify=y)
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled,y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)
decision_tree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)


In [29]:
def score_row(name, pred,scores=None):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    r = {
        "model": name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "found":int(tp),"missed":int(fn),"false_alarm":int(fp)}
    if scores is not None :
        r["auc"]= round(roc_auc_score(y_test,scores),4)
    return r
print("train",x_train.shape,"test:",x_test.shape)
print("diabetic patients int he test set:",int(np.sum(y_test)),"of",len(y_test))
for nm,m,xt in [("logistic regression",logreg,x_test_scaled),
                ("desicion tree (d=3)",decision_tree,x_test),
                ("KNN (k=25)",knn,x_test_scaled)]:
    print(f" {nm:22s}accuracy{accuracy_score(y_test,m.predict(xt)):.4f}")

train (760, 13) test: (190, 13)
diabetic patients int he test set: 55 of 190
 logistic regression   accuracy0.7526
 desicion tree (d=3)   accuracy0.7211
 KNN (k=25)            accuracy0.7474


## *Restate the target you are aiming*

In [30]:
TARGET_RECALL = 0.70
GUARD_F1 = 0.50
BEST_SO_FAR = 0.6182  # Week 5 Logistic Regression at threshold 0.30

print("PROJECT METRIC : recall  (guard: F1 >=", GUARD_F1, ")")
print("TARGET          : recall >=", TARGET_RECALL)
print("BEST SO FAR     :", BEST_SO_FAR, "(Week 5 baseline at threshold 0.30)")
print("GAP TO CLOSE    :", round(TARGET_RECALL - BEST_SO_FAR, 4))

PROJECT METRIC : recall  (guard: F1 >= 0.5 )
TARGET          : recall >= 0.7
BEST SO FAR     : 0.6182 (Week 5 baseline at threshold 0.30)
GAP TO CLOSE    : 0.0818


## *Example how an svm reaches a decision*
SVM uses three main concepts:

Hyperplane

A hyperplane is the decision boundary that separates different classes.

For example, it can separate Class A from Class B.

Margin

The margin is the distance between the hyperplane and the closest data points from each class.

SVM tries to find the hyperplane with the maximum margin.

A larger margin usually helps the model generalize better to new data.

Support Vectors

Support vectors are the data points closest to the hyperplane.

They are important because they determine the position of the hyperplane and the margin.

In [31]:
from sklearn.svm import SVC 
svm_rbf = SVC(random_state=42)
svm_rbf.fit(x_train_scaled,y_train)
pred_rbf = svm_rbf.predict(x_test_scaled)
print("fitted:",svm_rbf)
print("patients flagged:",int(pred_rbf.sum()),"of",len(pred_rbf))

fitted: SVC(random_state=42)
patients flagged: 17 of 190


In [32]:
row_rbf = score_row("SVC rfc (default)",pred_rbf,
                    svm_rbf.decision_function(x_test_scaled))
print()
print("TARGET RECALL:",TARGET_RECALL,"| best so far :",BEST_SO_FAR)
print("this model: ",row_rbf["recall"])                                       


TARGET RECALL: 0.7 | best so far : 0.6182
this model:  0.1818


## try a straight boundary instead

In [33]:
svm_linear = SVC(kernel="linear", random_state=42)
svm_linear.fit(x_train_scaled, y_train)

pred_linear = svm_linear.predict(x_test_scaled)

row_linear = score_row(
    "SVC Linear",
    pred_linear,
    svm_linear.decision_function(x_test_scaled)
)


print(pd.DataFrame([row_rbf, row_linear]).to_string(index=False))

print()
print("target recall:", TARGET_RECALL, "| best so far:", BEST_SO_FAR)
print("RBF recall   :", row_rbf["recall"])
print("Linear recall:", row_linear["recall"])

            model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
SVC rfc (default)    0.7263     0.5882  0.1818 0.2778     10      45            7 0.6443
       SVC Linear    0.7105     0.0000  0.0000 0.0000      0      55            0 0.6919

target recall: 0.7 | best so far: 0.6182
RBF recall   : 0.1818
Linear recall: 0.0


In [34]:
print("patirnts the linear svm flagged as diabetics :",
      int(pred_linear.sum()),"of",len(pred_linear))
print()
print("linear svm confution matrix")
print(confusion_matrix(y_test,pred_linear))
print("dummy model confution matrix")
print(confusion_matrix(y_test,np.zeros_like(y_test)))
print("accuracy score of linear svm :",
      round(accuracy_score(y_test,pred_linear),4))
print("accuracy score of dummy      :",
      round(accuracy_score(y_test,np.zeros_like(y_test)),4))

patirnts the linear svm flagged as diabetics : 0 of 190

linear svm confution matrix
[[135   0]
 [ 55   0]]
dummy model confution matrix
[[135   0]
 [ 55   0]]
accuracy score of linear svm : 0.7105
accuracy score of dummy      : 0.7105


*learn what class_wieght*

Class weight is a technique used in machine learning to give different levels of importance to different classes.

It is mainly useful when the dataset is imbalanced, where one class has many more samples than another.

Example
Suppose we have:

Class 0: 900 samples

Class 1: 100 samples

In [35]:
svm_rbf_bal = SVC(random_state=42, class_weight="balanced").fit(
    x_train_scaled, y_train
)

svm_lin_bal = SVC(kernel="linear", random_state=42,
                  class_weight="balanced").fit(
    x_train_scaled, y_train
)

rows = [
    row_rbf,
    score_row(
        "SVC rbf (balanced)",
        svm_rbf_bal.predict(x_test_scaled),
        svm_rbf_bal.decision_function(x_test_scaled)
    ),
    row_linear,
    score_row(
        "SVC linear (balanced)",
        svm_lin_bal.predict(x_test_scaled),
        svm_lin_bal.decision_function(x_test_scaled)
    )
]

svm_table = pd.DataFrame(rows)

print(svm_table.to_string(index=False))

                model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
    SVC rfc (default)    0.7263     0.5882  0.1818 0.2778     10      45            7 0.6443
   SVC rbf (balanced)    0.6421     0.4198  0.6182 0.5000     34      21           47 0.6826
           SVC Linear    0.7105     0.0000  0.0000 0.0000      0      55            0 0.6919
SVC linear (balanced)    0.6579     0.4375  0.6364 0.5185     35      20           45 0.7118


In [36]:
scores_bal= svm_rbf_bal.decision_function(x_test_scaled)
print("decision_function output for the first 8 test patients:")
print(np.round(scores_bal[:8], 3))

print("negative = healthy side of the boundary, positive = diabetic side")

print()

print("AUC from decision_function:", 
      round(roc_auc_score(y_test, scores_bal), 4))

print("AUC of the linear SVM that found nobody:",
      round(roc_auc_score(y_test, svm_linear.decision_function(x_test_scaled)), 4))

decision_function output for the first 8 test patients:
[ 0.027 -1.573 -0.752 -0.552 -0.74  -1.247  0.772  1.099]
negative = healthy side of the boundary, positive = diabetic side

AUC from decision_function: 0.6826
AUC of the linear SVM that found nobody: 0.6919


 *CHECK HOW MUCH OF THE DATA THE BOUNDARY RESTS ON*

In [37]:
print("support vectors per class (rbf default):", svm_rbf.n_support_)
print("total support vectors:", svm_rbf.n_support_.sum(), "of", len(y_train))
print(
    f"={(svm_rbf.n_support_.sum() / len(y_train)) * 100:.1f}% of the training set"
)

support vectors per class (rbf default): [277 212]
total support vectors: 489 of 760
=64.3% of the training set


In [39]:
svm_c10 = SVC(random_state=42, C=10).fit(x_train_scaled, y_train)

print(pd.DataFrame([
    row_rbf,
    score_row(
        "SVC rbf C=10",
        svm_c10.predict(x_test_scaled),
        svm_c10.decision_function(x_test_scaled)
    )
]).to_string(index=False))

            model  accuracy  precision  recall     f1  found  missed  false_alarm    auc
SVC rfc (default)    0.7263     0.5882  0.1818 0.2778     10      45            7 0.6443
     SVC rbf C=10    0.6684     0.3947  0.2727 0.3226     15      40           23 0.6221
